# TP 3 — Contours et reconnaissance de motifs

**Objectifs**

- Calculer un gradient et une magnitude de Sobel à la main avec NumPy.
- Comparer Sobel et Canny pour la détection de bords.
- Détecter des lignes droites avec la transformation de Hough probabiliste.
- Localiser un motif dans une image complexe via *template matching*.

**Durée indicative : 50 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import patches
from scipy import ndimage as ndi
from skimage import color, data, feature, filters, transform, util

img = data.camera()  # niveaux de gris uint8
img_f = util.img_as_float(img)

## Exercice 1 — Sobel à la main

1. Définissez les noyaux de Sobel `kx` et `ky` (matrices 3×3).
2. Convoluez l'image en niveaux de gris avec chacun de ces noyaux à l'aide de `scipy.ndimage.convolve`.
3. Calculez la **magnitude** du gradient : `mag = sqrt(gx² + gy²)`.
4. Affichez `gx`, `gy` et `mag` côte à côte (utilisez `cmap="gray"`).

<details>
<summary>💡 Coup de pouce</summary>

- Noyaux : `kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])` pour les bords verticaux. `ky = kx.T` (transposée) pour les bords horizontaux.
- Convoluez après cast en float : `gx = ndi.convolve(img.astype(float), kx)`.
- Magnitude : `mag = np.sqrt(gx ** 2 + gy ** 2)`.

</details>

In [2]:
# TODO

## Exercice 2 — Sobel vs Canny

1. Calculez la magnitude de Sobel via `skimage.filters.sobel`.
2. Calculez les contours de Canny via `skimage.feature.canny` avec `sigma=2`, `low_threshold=0.1`, `high_threshold=0.2`.
3. Affichez l'originale, le Sobel, et le Canny. Commentez : pourquoi le Canny donne-t-il des contours fins là où le Sobel donne une carte dense ?

<details>
<summary>💡 Coup de pouce</summary>

- `filters.sobel` renvoie une **image float** dans [0, 1] : pas besoin de calculer la magnitude à la main.
- `feature.canny` attend une image **float** : convertissez avec `util.img_as_float` si vous partez de uint8.
- Canny gomme les contours faibles isolés via l'hystérésis : c'est ce qui explique la différence visuelle vs Sobel.

</details>

In [3]:
# TODO

## Exercice 3 — Détection de lignes par Hough

On utilise une image avec des lignes claires (`data.checkerboard` ou une image synthétique).

1. Construisez une image avec quelques lignes (utilisez `np.zeros` puis tracez 3-4 lignes via `skimage.draw.line`).
2. Ajoutez un peu de bruit gaussien.
3. Calculez les contours avec Canny.
4. Appliquez `transform.probabilistic_hough_line` pour récupérer les segments détectés.
5. Tracez l'image originale et superposez les segments détectés (en rouge).

<details>
<summary>💡 Coup de pouce</summary>

- Pour tracer une ligne : `from skimage.draw import line; rr, cc = line(y0, x0, y1, x1); img[rr, cc] = 255`.
- Le bruit gaussien sur image binaire : `img + np.random.normal(0, 5, img.shape)` puis cliper.
- `probabilistic_hough_line(edges, threshold=10, line_length=30, line_gap=3)` retourne une **liste** de segments `[((x0, y0), (x1, y1)), ...]`.
- Tracé : pour chaque segment, `ax.plot([x0, x1], [y0, y1], 'r-')`.

</details>

In [4]:
# TODO — créer image avec lignes et la traiter

## Exercice 4 — Template matching

1. Récupérez l'image `data.coins()` et extrayez la pièce située dans la zone `[140:200, 220:280]`. Cette extraction sert de **template**.
2. Avec `feature.match_template`, calculez la carte de corrélation entre `coins` et le template.
3. Trouvez la position du **meilleur match** via `np.unravel_index`.
4. Affichez l'image, le template, la carte de corrélation, et tracez un rectangle (avec `matplotlib.patches.Rectangle`) autour de la position détectée.

**Bonus** : trouvez les 3 meilleures positions (locales) en seuillant la carte de corrélation. Vous pouvez utiliser `feature.peak_local_max`.

<details>
<summary>💡 Coup de pouce</summary>

- Extraction du template : `template = img[140:200, 220:280]` (= une pièce).
- `result = match_template(img, template)` : la **carte de corrélation** est plus petite que l'image (de la taille du template).
- Position du max : `y, x = np.unravel_index(np.argmax(result), result.shape)` (renvoie le **coin haut-gauche** du match, pas le centre).
- Rectangle : `Rectangle((x, y), template.shape[1], template.shape[0])` (matplotlib veut `(x, y)`).
- **Bonus** : `peak_local_max(result, min_distance=20, threshold_rel=0.7)` retourne tous les pics.

</details>

In [5]:
# TODO